## 0. Conexión

In [1]:
from pathlib import Path
import duckdb

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "analytics_engineer_assets").exists())
DB = ROOT / "warehouse.duckdb"

con = duckdb.connect(str(DB), read_only=True)

def q(sql):
    """Ejecuta una consulta y muestra el resultado como tabla."""
    con.sql(sql).show(max_rows=50)

q("SELECT table_schema, table_name FROM information_schema.tables ORDER BY 1, 2")

┌──────────────┬─────────────┐
│ table_schema │ table_name  │
│   varchar    │   varchar   │
├──────────────┼─────────────┤
│ raw          │ customers   │
│ raw          │ fx_rates    │
│ raw          │ order_items │
│ raw          │ orders      │
│ raw          │ products    │
└──────────────┴─────────────┘



## 1. Vista general de cada tabla

In [ ]:
q("SELECT * FROM raw.orders LIMIT 10")

In [ ]:
q("SELECT * FROM raw.order_items LIMIT 10")

In [ ]:
q("SELECT id, name, category, base_price, currency FROM raw.products LIMIT 10")

In [ ]:
q("SELECT * FROM raw.customers LIMIT 10")

In [ ]:
q("SELECT * FROM raw.fx_rates")

### Volumen y rango de fechas

In [2]:
q("""
    SELECT 'customers' AS tabla, count(*) AS filas, count(DISTINCT id) AS ids_distintos FROM raw.customers
    UNION ALL SELECT 'orders',      count(*), count(DISTINCT id) FROM raw.orders
    UNION ALL SELECT 'order_items', count(*), count(DISTINCT id) FROM raw.order_items
    UNION ALL SELECT 'products',    count(*), count(DISTINCT id) FROM raw.products
""")

q("""
    SELECT min(order_date::TIMESTAMP) AS primera_orden,
           max(order_date::TIMESTAMP) AS ultima_orden
    FROM raw.orders
""")

┌─────────────┬───────┬───────────────┐
│    tabla    │ filas │ ids_distintos │
│   varchar   │ int64 │     int64     │
├─────────────┼───────┼───────────────┤
│ customers   │    50 │            50 │
│ orders      │   453 │           453 │
│ order_items │   363 │           363 │
│ products    │   100 │           100 │
└─────────────┴───────┴───────────────┘

┌─────────────────────┬─────────────────────┐
│    primera_orden    │    ultima_orden     │
│      timestamp      │      timestamp      │
├─────────────────────┼─────────────────────┤
│ 2024-01-16 14:30:00 │ 2024-12-31 15:55:00 │
└─────────────────────┴─────────────────────┘



### Status de las órdenes

Todas están `completed`. Igual se testea con `accepted_values` para detectar valores nuevos en el futuro.

In [ ]:
q("SELECT status, count(*) AS ordenes FROM raw.orders GROUP BY status")

## 2. Hallazgo: monedas inválidas

`XYZ`, `ABC` y `QWE` no son códigos ISO 4217. Aparecen tanto en órdenes como en ítems.

**Decisión:** se conservan las filas, con `revenue_usd` nulo y un flag de moneda inválida. Cuentan para unidades vendidas, pero no para revenue.

In [3]:
q("""
    SELECT currency, count(*) AS ordenes,
           round(100.0 * count(*) / sum(count(*)) OVER (), 1) AS pct
    FROM raw.orders
    GROUP BY currency
    ORDER BY ordenes DESC
""")

┌──────────┬─────────┬────────┐
│ currency │ ordenes │  pct   │
│ varchar  │  int64  │ double │
├──────────┼─────────┼────────┤
│ USD      │     244 │   53.9 │
│ EUR      │     126 │   27.8 │
│ XYZ      │      27 │    6.0 │
│ ABC      │      27 │    6.0 │
│ QWE      │      25 │    5.5 │
│ GBP      │       4 │    0.9 │
└──────────┴─────────┴────────┘



In [ ]:
q("""
    SELECT currency, count(*) AS items,
           round(100.0 * count(*) / sum(count(*)) OVER (), 1) AS pct
    FROM raw.order_items
    GROUP BY currency
    ORDER BY items DESC
""")

## 3. Hallazgo: productos fuera del catálogo

`order_items.product_id` no es una FK garantizada. Los IDs 101 a 110 no existen en `products.json`, y todas esas líneas son de agosto en adelante.

**Decisión:** no se descartan. Se crea un miembro "producto desconocido" en `dim_products` por cada ID huérfano, con `is_in_catalogue = false`.

In [ ]:
q("""
    SELECT oi.product_id, count(*) AS lineas
    FROM raw.order_items oi
    LEFT JOIN raw.products p ON p.id = oi.product_id::INT
    WHERE p.id IS NULL
    GROUP BY oi.product_id
    ORDER BY oi.product_id::INT
""")

In [ ]:
q("""
    SELECT CASE WHEN p.id IS NULL THEN 'fuera de catálogo' ELSE 'en catálogo' END AS estado,
           count(*) AS lineas,
           min(o.order_date::TIMESTAMP) AS desde,
           max(o.order_date::TIMESTAMP) AS hasta
    FROM raw.order_items oi
    JOIN raw.orders o ON o.id = oi.order_id
    LEFT JOIN raw.products p ON p.id = oi.product_id::INT
    GROUP BY estado
""")

## 4. Hallazgo: órdenes sin líneas de ítems

Las órdenes con ID 361 a 453 no tienen ninguna línea en `order_items`.

**Decisión:** quedan fuera del análisis por producto (no sabemos qué se vendió), pero sí cuentan para el análisis por hora del día.

In [ ]:
q("""
    SELECT count(*) AS ordenes_sin_items,
           min(o.id::INT) AS id_desde,
           max(o.id::INT) AS id_hasta
    FROM raw.orders o
    WHERE NOT EXISTS (SELECT 1 FROM raw.order_items oi WHERE oi.order_id = o.id)
""")

## 5. Hallazgo: moneda de la orden distinta a la del ítem

**Decisión:** el revenue de cada línea se calcula con la moneda de la línea.

In [ ]:
q("""
    SELECT o.currency AS moneda_orden, oi.currency AS moneda_item, count(*) AS lineas
    FROM raw.orders o
    JOIN raw.order_items oi ON oi.order_id = o.id
    WHERE o.currency <> oi.currency
    GROUP BY ALL
    ORDER BY lineas DESC
""")

## 6. Hallazgo: total de la orden vs. suma de sus líneas

Muchas diferencias son de centavos (redondeo de `quantity × unit_price`), pero otras son grandes.

**Decisión:** el revenue por producto sale de las líneas. Un test con tolerancia distingue el redondeo de las diferencias reales.

In [ ]:
q("""
    WITH comparacion AS (
        SELECT o.id,
               o.total_amount::DOUBLE AS total_orden,
               sum(oi.quantity::INT * oi.unit_price::DOUBLE) AS suma_lineas
        FROM raw.orders o
        JOIN raw.order_items oi ON oi.order_id = o.id
        GROUP BY o.id, o.total_amount
    )
    SELECT CASE
             WHEN abs(total_orden - suma_lineas) <= 0.05 THEN '1) coincide o redondeo (<= 0.05)'
             WHEN abs(total_orden - suma_lineas) <= 1    THEN '2) diferencia chica (<= 1)'
             ELSE                                             '3) diferencia real (> 1)'
           END AS categoria,
           count(*) AS ordenes
    FROM comparacion
    GROUP BY categoria
    ORDER BY categoria
""")

In [ ]:
q("""
    SELECT o.id,
           o.total_amount::DOUBLE AS total_orden,
           round(sum(oi.quantity::INT * oi.unit_price::DOUBLE), 2) AS suma_lineas
    FROM raw.orders o
    JOIN raw.order_items oi ON oi.order_id = o.id
    GROUP BY o.id, o.total_amount
    HAVING abs(total_orden - suma_lineas) > 1
    ORDER BY o.id::INT
    LIMIT 10
""")

## 7. Hallazgo: precio unitario vs. precio de catálogo

En la mayoría de las líneas `unit_price` es igual al `base_price` del catálogo (en USD), **aunque la línea diga EUR o GBP**. Eso sugiere que la etiqueta de moneda en origen podría no ser confiable.

**Decisión:** no se corrige (no podemos saberlo con certeza), pero se documenta como riesgo.

In [ ]:
q("""
    SELECT oi.currency AS moneda_linea,
           count(*) AS lineas_en_catalogo,
           sum(CASE WHEN oi.unit_price::DOUBLE = p.base_price THEN 1 ELSE 0 END) AS precio_igual_catalogo
    FROM raw.order_items oi
    JOIN raw.products p ON p.id = oi.product_id::INT
    GROUP BY oi.currency
    ORDER BY lineas_en_catalogo DESC
""")

## 8. Tasas de cambio

`fx_rates.json` solo trae dos fechas (2024-06-01 y 2024-09-15), pero hay órdenes desde enero. Además, GBP como moneda base solo aparece en junio.

**Decisiones:**
- Cada orden toma la última tasa anterior o igual a su fecha (join por rango de vigencia o `ASOF JOIN`).
- Las órdenes previas a 2024-06-01 usan la primera tasa disponible y quedan marcadas con un flag.
- Si falta el par directo (ej. GBP→USD en septiembre), se usa la inversa de USD→GBP.

In [ ]:
q("""
    SELECT base_currency,
           rate_date::DATE AS rate_date,
           k AS moneda_destino,
           (rates ->> k)::DOUBLE AS tasa
    FROM raw.fx_rates, unnest(json_keys(rates)) AS t(k)
    ORDER BY rate_date, base_currency, moneda_destino
""")

In [ ]:
q("""
    SELECT CASE WHEN order_date::DATE < DATE '2024-06-01' THEN 'antes de la primera tasa'
                ELSE 'con tasa disponible' END AS cobertura,
           count(*) AS ordenes
    FROM raw.orders
    GROUP BY cobertura
""")

## 9. Patrones temporales (anticipo de la pregunta de negocio 2)

- Solo hay órdenes entre las 9 y las 18 h: parece horario local de negocio. La zona horaria no está documentada y es un supuesto a declarar.
- El volumen salta en agosto (de ~13 a ~77 órdenes por mes). Hay que tenerlo en cuenta al interpretar promedios.

In [ ]:
q("""
    SELECT hour(order_date::TIMESTAMP) AS hora, count(*) AS ordenes
    FROM raw.orders
    GROUP BY hora
    ORDER BY hora
""")

In [ ]:
q("""
    SELECT dayname(order_date::TIMESTAMP) AS dia,
           isodow(order_date::TIMESTAMP)  AS n_dia,
           count(*) AS ordenes
    FROM raw.orders
    GROUP BY ALL
    ORDER BY n_dia
""")

In [ ]:
q("""
    SELECT strftime(date_trunc('month', order_date::TIMESTAMP), '%Y-%m') AS mes,
           count(*) AS ordenes
    FROM raw.orders
    GROUP BY mes
    ORDER BY mes
""")

## 10. Integridad referencial restante

In [ ]:
q("""
    SELECT 'orders con customer inexistente' AS chequeo, count(*) AS filas
    FROM raw.orders o
    WHERE NOT EXISTS (SELECT 1 FROM raw.customers c WHERE c.id = o.customer_id)
    UNION ALL
    SELECT 'order_items con order inexistente', count(*)
    FROM raw.order_items oi
    WHERE NOT EXISTS (SELECT 1 FROM raw.orders o WHERE o.id = oi.order_id)
    UNION ALL
    SELECT 'order_items con quantity <= 0', count(*)
    FROM raw.order_items WHERE quantity::INT <= 0
    UNION ALL
    SELECT 'order_items con unit_price <= 0', count(*)
    FROM raw.order_items WHERE unit_price::DOUBLE <= 0
""")

## 11. Resumen de calidad de datos

Tabla consolidada para copiar en `design_notes.md`.

In [ ]:
q("""
    WITH valid AS (SELECT unnest(['USD', 'EUR', 'GBP']) AS c),
    line_totals AS (
        SELECT order_id, sum(quantity::INT * unit_price::DOUBLE) AS s
        FROM raw.order_items GROUP BY order_id
    )
    SELECT 'Órdenes con moneda inválida' AS problema,
           count(*) FILTER (WHERE currency NOT IN (SELECT c FROM valid)) AS afectadas,
           count(*) AS total
    FROM raw.orders
    UNION ALL
    SELECT 'Ítems con moneda inválida',
           count(*) FILTER (WHERE currency NOT IN (SELECT c FROM valid)), count(*)
    FROM raw.order_items
    UNION ALL
    SELECT 'Ítems con producto fuera de catálogo',
           count(*) FILTER (WHERE product_id::INT NOT IN (SELECT id FROM raw.products)), count(*)
    FROM raw.order_items
    UNION ALL
    SELECT 'Órdenes sin ítems',
           count(*) FILTER (WHERE id NOT IN (SELECT order_id FROM raw.order_items)), count(*)
    FROM raw.orders
    UNION ALL
    SELECT 'Ítems con moneda distinta a su orden',
           count(*) FILTER (WHERE oi.currency <> o.currency), count(*)
    FROM raw.order_items oi JOIN raw.orders o ON o.id = oi.order_id
    UNION ALL
    SELECT 'Órdenes cuyo total difiere de sus líneas (> 1)',
           count(*) FILTER (WHERE abs(o.total_amount::DOUBLE - lt.s) > 1), count(*)
    FROM raw.orders o JOIN line_totals lt ON lt.order_id = o.id
    UNION ALL
    SELECT 'Órdenes anteriores a la primera tasa FX',
           count(*) FILTER (WHERE order_date::DATE < DATE '2024-06-01'), count(*)
    FROM raw.orders
""")

## 12. Cerrar la conexión

Cerrarla antes de ejecutar dbt, para evitar errores de *lock*.

In [ ]:
con.close()
print("Conexión cerrada")